# Лабораторная работа 4. Многомерная бинарная классификация

Датасет: `datasets/student.csv`. Предсказываем `G3`.

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [21]:
df = pd.read_csv('student.csv')
df = df.drop('Unnamed: 0', axis=1)
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,no,no,4,3,4,1,1,3,6,0
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,yes,no,5,3,3,1,1,3,4,0
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,yes,no,4,3,2,2,3,3,10,0
3,GP,F,15,U,GT3,T,4,2,health,services,...,yes,yes,3,2,2,1,1,5,2,1
4,GP,F,16,U,GT3,T,3,3,other,other,...,no,no,4,3,2,1,2,5,4,0


# Задание 1
Предобработайте данные. Для числовых признаков используйте стандартизацию, для категориальных - one-hot-кодирование. Разделите выборку на обучающую и тестовую. Размер выборок выберите самостоятельно.

In [22]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

X = df.drop('G3', axis=1)
y = df['G3']

num_cols = [
    "age", "Medu", "Fedu", "traveltime", "studytime", "failures",
    "famrel", "freetime", "goout", "Dalc", "Walc", "health", "absences"
]
cat_cols = [
    "school", "sex", "address", "famsize", "Pstatus",
    "Mjob", "Fjob", "reason", "guardian",
    "schoolsup", "famsup", "paid", "activities", "nursery",
    "higher", "internet", "romantic"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

In [23]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_prep = pd.DataFrame(
    X_train_prep,
    columns=feature_names,
    index=X_train.index
)

X_test_prep = pd.DataFrame(
    X_test_prep,
    columns=feature_names,
    index=X_test.index
)

In [24]:
X_train_prep.head()

,num__age,num__Medu,num__Fedu,num__traveltime,num__studytime,num__failures,num__famrel,num__freetime,num__goout,num__Dalc,...,cat__activities_no,cat__activities_yes,cat__nursery_no,cat__nursery_yes,cat__higher_no,cat__higher_yes,cat__internet_no,cat__internet_yes,cat__romantic_no,cat__romantic_yes
181,-0.586396,0.246437,0.423207,-0.625779,-0.056853,-0.456744,0.064432,-1.195981,-0.144396,-0.55443,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0
194,-0.586396,-0.680636,0.423207,0.828236,-1.254553,-0.456744,1.195575,-0.214345,-0.144396,-0.55443,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
173,-0.586396,-1.607709,0.423207,-0.625779,-0.056853,3.628097,0.064432,-0.214345,1.644982,-0.55443,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
63,-0.586396,1.173510,0.423207,-0.625779,1.140847,-0.456744,-1.066710,0.767292,0.750293,0.55443,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
253,-0.586396,-0.680636,-1.434203,0.828236,-1.254553,-0.456744,-1.066710,-0.214345,-1.039085,-0.55443,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0


# Задание 2

Обучите следующие модели:
1. SVM
2. Логистическая регрессия
3. KNN
4. Дерево решений

Подберите оптимальные гиперпараметры с помощью кросс-валидации

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "SVM": {
        "model": SVC(random_state=42),
        "params": {
            "clf__C": [0.1, 1, 10],
            "clf__gamma": ["scale", 0.01, 0.1],
            "clf__kernel": ["rbf", "linear"]
        }
    },
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=5000, random_state=42),
        "params": {
            "clf__C": [0.01, 0.1, 1, 10],
            "clf__solver": ["lbfgs", "liblinear"]
        }
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "clf__n_neighbors": [3, 5, 7, 9, 11],
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["euclidean", "manhattan"]
        }
    },
    "DecisionTree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "clf__criterion": ["gini", "entropy"],
            "clf__max_depth": [3, 5, 7, 10, None],
            "clf__min_samples_split": [2, 5, 10],
            "clf__min_samples_leaf": [1, 2, 4]
        }
    }
}

results = {}

for name, config in models.items():
    print(f"\n{'='*60}")
    print(f"Модель: {name}")
    print(f"{'='*60}")

    pipe = Pipeline([
        ("prep", preprocessor),
        ("clf", config["model"])
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=config["params"],
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    results[name] = {
        "best_params": grid.best_params_,
        "best_cv_score": grid.best_score_,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_report": classification_report(y_test, y_pred),
        "confusion_matrix": confusion_matrix(y_test, y_pred)
    }

    print(f"Лучшие параметры: {grid.best_params_}")
    print(f"Лучший f1_macro (CV): {grid.best_score_:.4f}")
    print(f"Accuracy на тесте: {accuracy_score(y_test, y_pred):.4f}")
    print("Classification report:")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))


Модель: SVM
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Лучшие параметры: {'clf__C': 1, 'clf__gamma': 'scale', 'clf__kernel': 'linear'}
Лучший f1_macro (CV): 0.6636
Accuracy на тесте: 0.6709
Classification report:
              precision    recall  f1-score   support

           0       0.71      0.53      0.61        38
           1       0.65      0.80      0.72        41

    accuracy                           0.67        79
   macro avg       0.68      0.67      0.66        79
weighted avg       0.68      0.67      0.66        79

Confusion matrix:
[[20 18]
 [ 8 33]]

Модель: LogisticRegression
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Лучшие параметры: {'clf__C': 10, 'clf__solver': 'lbfgs'}
Лучший f1_macro (CV): 0.6568
Accuracy на тесте: 0.7089
Classification report:
              precision    recall  f1-score   support

           0       0.78      0.55      0.65        38
           1       0.67      0.85      0.75        41

    accuracy     

In [27]:
print("\n" + "="*60)
print("ИТОГОВАЯ СВОДКА")
print("="*60)
summary = pd.DataFrame({
    name: {
        "CV f1_macro": res["best_cv_score"],
        "Test accuracy": res["test_accuracy"],
        "Best params": res["best_params"]
    }
    for name, res in results.items()
}).T

print(summary)


ИТОГОВАЯ СВОДКА
                   CV f1_macro Test accuracy  \
SVM                   0.663637      0.670886   
LogisticRegression    0.656787      0.708861   
KNN                    0.60687       0.56962   
DecisionTree          0.637372      0.582278   

                                                          Best params  
SVM                 {'clf__C': 1, 'clf__gamma': 'scale', 'clf__ker...  
LogisticRegression             {'clf__C': 10, 'clf__solver': 'lbfgs'}  
KNN                 {'clf__metric': 'euclidean', 'clf__n_neighbors...  
DecisionTree        {'clf__criterion': 'entropy', 'clf__max_depth'...  


# Задание 3

Для каждой модели посчитайте Accuracy, Precision, Recall, F1-score. Постройте confusion matrix и roc-кривую.

# Задание 4

Для логистической регрессии выберите оптимальный порог классификации. Постройте график зависимости метрик от порога классификации. Сравнить качество модели при оптимальном пороге и пороге равном 0.5.